In [0]:
from pathlib import Path

github_root = Path(
    "/Workspace/Users/prxaura@gmail.com/materials-platform/lammps"
)

runs_dir = github_root / "Si" / "runs"

print(runs_dir)
print(runs_dir.exists())

In [0]:
import json

records = []

for run_dir in runs_dir.iterdir():

    if not run_dir.is_dir():
        continue

    run_file = run_dir / "run.json"

    if not run_file.exists():
        continue

    with open(run_file) as f:
        data = json.load(f)
    print(data)
    data["run_id"] = run_dir.name
    records.append(data)

df = spark.createDataFrame(records)

display(df)

In [0]:
from pathlib import Path
from pyspark.sql import functions as F

# Collect all lattice-parameter log files
bulk_modulus_log_files = []

for run_dir in runs_dir.iterdir():

    if not run_dir.is_dir():
        continue

    lp_dir = run_dir / "results" / "bulk_modulus"

    if not lp_dir.exists():
        continue

    files = list(lp_dir.rglob("*.txt"))

    bulk_modulus_log_files.extend(files)


print(f"Found {len(bulk_modulus_log_files)} log files")


# Read all log files into ONE Spark DataFrame
raw_dfs = []

for f in bulk_modulus_log_files:

    df = (
        spark.read
        .text(str(f))
        .withColumn("source_file", F.lit(str(f)))
    )

    raw_dfs.append(df)


raw_df = raw_dfs[0]

for df in raw_dfs[1:]:
    raw_df = raw_df.unionByName(df)

display(raw_df)

In [0]:
e_vs_v_df = (
    raw_df
    .filter(
        F.col("value").rlike(
            r"^\s*[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?\s+"
            r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?\s+"
            r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?\s+"
            r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?\s*$"
        )
    )
    .select(
        F.split(F.trim(F.col("value")), r"\s+").alias("cols"),
        "source_file"
    )
    .select(
        F.col("cols")[0].cast("double").alias("lattice_parameter"),
        F.col("cols")[1].cast("double").alias("volume"),
        F.col("cols")[2].cast("double").alias("energy"),
        F.col("cols")[3].cast("double").alias("pressure"),
        "source_file"
    )
)

display(e_vs_v_df)

In [0]:
%sql

drop table if exists workspace.B_silicon.EV_raw

In [0]:
%sql
create table if not exists workspace.B_silicon.EV_raw

In [0]:
display(e_vs_v_df)

In [0]:
e_vs_v_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.B_silicon.EV_raw"
    )

In [0]:
%sql

select * from B_silicon.EV_raw

In [0]:
%sql

select * from B_silicon.lattice_parameter_simulation_runs;